# Contested Norms
## Extract message status from logs

In [ ]:
from collections import Counter
from csv import DictReader, DictWriter
import os
import re

In [ ]:
data_dir = "/cs/tresorit/Reddit Contested Norms"
active_file = "transformed/2025-12-09-2ssp3-active_sample-accounts.tsv"
banned_file = "transformed/2025-12-09-2ssp3-ban_sample-accounts.tsv"
removed_file = "transformed/2025-12-09-2ssp3-removal_sample-accounts.tsv"

log_files = [
    "interventions/2025-12-09-2ssp3-distribute_survey.log",
    "interventions/2025-12-10-2ssp3-distribute_survey.log",
    "interventions/2025-12-10-2ssp3-distribute_survey-002.log",
    "interventions/2025-12-10-2ssp3-distribute_survey-003.log",
    "interventions/2025-12-10-2ssp3-distribute_survey-004.log"]

reminder_log_files = [
    "interventions/2026-01-20-2ssp3-active_sample-distribute_survey.log",
    "interventions/2026-01-20-2ssp3-ban_sample-distribute_survey.log",
    "interventions/2026-01-20-2ssp3-removal_sample-distribute_survey.log"
]

### Original messaging

In [ ]:
active = []
banned = []
removed = []
with open(os.path.join(data_dir, active_file)) as f:
    reader = DictReader(f, delimiter='\t')
    for row in reader:
        active.append(row)
with open(os.path.join(data_dir, banned_file)) as f:
    reader = DictReader(f, delimiter='\t')
    for row in reader:
        banned.append(row)
with open(os.path.join(data_dir, removed_file)) as f:
    reader = DictReader(f, delimiter='\t')
    for row in reader:
        removed.append(row)

In [ ]:
active_users = set(row['username'] for row in active)
banned_users = set(row['username'] for row in banned)
removed_users = set(row['username'] for row in removed)

In [ ]:
re_send = re.compile(r".*Sending a message to user (.+) with data")
re_survey_url = re.compile(r".*'survey_url': '(.+)'")
re_unknown = re.compile(r".*User not found: (\S+)")
re_failed = re.compile(r".*Failed to send message to (\S+)")
re_success = re.compile(r".*Message successfully sent to user (\S+)")
re_500 = re.compile(
    r".*500 Server Error: Internal Server Error for url: "
    "https://oauth.reddit.com/user/(.*)/about/")

### Reminder messages

In [ ]:
unknown = set()
failed = set()
urls = {}
success = set()
for log_file in log_files:
    with open(os.path.join(data_dir, log_file), 'r') as f:
        for row in f:
            if re_send.match(row):
                user = re_send.match(row).groups()[0]
                url = re_survey_url.match(row).groups()[0]
                urls[user] = url
            elif re_unknown.match(row):
                unknown.add(re_unknown.match(row).groups()[0])
            elif re_failed.match(row):
                failed.add(re_failed.match(row).groups()[0])
            elif re_500.match(row):
                failed.add(re_500.match(row).groups()[0])
            elif re_success.match(row):
                success.add(re_success.match(row).groups()[0])

In [ ]:
print("Active")
print("  {} Total".format(len(active_users)))
print("  {} Successful".format(len(active_users & success)))
print("  {} Unknown".format(len(active_users & unknown)))
print("  {} Failed".format(len(active_users & failed)))
print("  {} Unaccounted".format(
    len(active_users - success - unknown - failed)))

print("Banned")
print("  {} Total".format(len(banned_users)))
print("  {} Successful".format(len(banned_users & success)))
print("  {} Unknown".format(len(banned_users & unknown)))
print("  {} Failed".format(len(banned_users & failed)))
print("  {} Unaccounted".format(
    len(banned_users - success - unknown - failed)))

print("Removed")
print("  {} Total".format(len(removed_users)))
print("  {} Successful".format(len(removed_users & success)))
print("  {} Unknown".format(len(removed_users & unknown)))
print("  {} Failed".format(len(removed_users & failed)))
print("  {} Unaccounted".format(
    len(removed_users - success - unknown - failed)))
print(list(removed_users - success - unknown - failed))

In [ ]:
active_messaged = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in active_users)

removed_messaged = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in removed_users)

banned_messaged = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in banned_users)

In [ ]:
unknown = set()
failed = set()
urls = {}
success = set()
for log_file in reminder_log_files:
    with open(os.path.join(data_dir, log_file), 'r') as f:
        for row in f:
            if re_send.match(row):
                user = re_send.match(row).groups()[0]
                url = re_survey_url.match(row).groups()[0]
                urls[user] = url
            elif re_unknown.match(row):
                unknown.add(re_unknown.match(row).groups()[0])
            elif re_failed.match(row):
                failed.add(re_failed.match(row).groups()[0])
            elif re_500.match(row):
                failed.add(re_500.match(row).groups()[0])
            elif re_success.match(row):
                success.add(re_success.match(row).groups()[0])

In [ ]:
print("Active")
print("  {} Total".format(len(active_users)))
print("  {} Successful".format(len(active_users & success)))
print("  {} Unknown".format(len(active_users & unknown)))
print("  {} Failed".format(len(active_users & failed)))
print("  {} Unaccounted".format(
    len(active_users - success - unknown - failed)))

print("Banned")
print("  {} Total".format(len(banned_users)))
print("  {} Successful".format(len(banned_users & success)))
print("  {} Unknown".format(len(banned_users & unknown)))
print("  {} Failed".format(len(banned_users & failed)))
print("  {} Unaccounted".format(
    len(banned_users - success - unknown - failed)))

print("Removed")
print("  {} Total".format(len(removed_users)))
print("  {} Successful".format(len(removed_users & success)))
print("  {} Unknown".format(len(removed_users & unknown)))
print("  {} Failed".format(len(removed_users & failed)))
print("  {} Unaccounted".format(
    len(removed_users - success - unknown - failed)))
print(list(removed_users - success - unknown - failed))

In [ ]:
active_reminded = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in active_users)

removed_reminded = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in removed_users)

banned_reminded = dict(
    (user, 1) if user in success
    else (user, 0)
    for user in banned_users)

In [ ]:
with open("2026-02-03-active_sample-message_status.csv", 'w', encoding="utf-8") as f:
    writer = DictWriter(f, ["username", "messaged", "reminded"])
    writer.writeheader()
    for user in active_users:
        writer.writerow({
            "username": user,
            "messaged": active_messaged[user],
            "reminded": active_reminded[user]
        })

with open("2026-02-03-ban_sample-message_status.csv", 'w', encoding="utf-8") as f:
    writer = DictWriter(f, ["username", "messaged", "reminded"])
    writer.writeheader()
    for user in banned_users:
        writer.writerow({
            "username": user,
            "messaged": banned_messaged[user],
            "reminded": banned_reminded[user]
        })

with open("2026-02-03-removal_sample-message_status.csv", 'w', encoding="utf-8") as f:
    writer = DictWriter(f, ["username", "messaged", "reminded"])
    writer.writeheader()
    for user in removed_users:
        writer.writerow({
            "username": user,
            "messaged": removed_messaged[user],
            "reminded": removed_reminded[user]
        })